In [1]:
%pip install minio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 2.8 MB/s eta 0:00:00
  Using cached pycryptodome-3.23.0-cp37-abi3-macosx_10_9_universal2.whl (2.5 MB)

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
MINIO_HOST = 'localhost'
MINIO_ACCESS_KEY = 'admin'
MINIO_SECRET_KEY = 'admin1234'

In [3]:
from minio import Minio

minio_client = Minio(
    f"{MINIO_HOST}:9000",
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

In [4]:
minio_client.bucket_exists('user-pics')

False

In [33]:
if not minio_client.bucket_exists('user-pics'):
    minio_client.make_bucket('user-pics')

In [6]:
minio_client.bucket_exists('user-pics')

True

In [7]:
minio_client.list_buckets()

[Bucket(name='user-pics', creation_date=datetime.datetime(2025, 10, 11, 10, 53, 21, 327000, tzinfo=datetime.timezone.utc))]

---

In [8]:
import io

In [9]:
with open('cat.jpg', 'rb') as f:
    data = io.BytesIO(f.read())

result = minio_client.put_object(
    bucket_name='user-pics',
    object_name='cat.jpg',
    data=data,
    length=-1,
    part_size=5*1024*1024
)
print(
    f'created {result.object_name} object; etag: {result.etag}, version-id: {result.version_id}'
)

created cat.jpg object; etag: 9563277ca4f0c8833911b2b70eb3fcec, version-id: None


In [10]:
data.seek(0)

0

In [11]:
result = minio_client.put_object(
    bucket_name='user-pics',
    object_name='data/cat_meta.jpg',
    data=data,
    length=-1,
    part_size=5*1024*1024,
    metadata={'creationPlace': 'Moscow, Russia'}
)
print(
    f'created {result.object_name} object; etag: {result.etag}, version-id: {result.version_id}'
)

created data/cat_meta.jpg object; etag: 9563277ca4f0c8833911b2b70eb3fcec, version-id: None


In [12]:
from datetime import datetime, timedelta
from minio.retention import Retention
from minio.commonconfig import GOVERNANCE, Tags

In [13]:
data.seek(0)

0

In [14]:
# date = datetime.utcnow().replace(
#     hour=0, minute=0, second=0, microsecond=0,
# ) + timedelta(days=30)

tags = Tags(for_object=True)
tags['usergroup'] = 'teacher'

result = minio_client.put_object(
    bucket_name='user-pics',
    object_name='cat_tags.jpg',
    data=data,
    length=-1,
    part_size=5*1024*1024,
    metadata={'creationPlace': 'Moscow, Russia'},
    tags=tags,
)

print(
    f'created {result.object_name} object; etag: {result.etag}, version-id: {result.version_id}'
)

created cat_tags.jpg object; etag: 9563277ca4f0c8833911b2b70eb3fcec, version-id: None


---

In [17]:
import json

<img src="http://localhost:9000/user-pics/cat.jpg"></img>

In [ ]:
http://localhost:9000/user-pics/cat.jpg

In [22]:
minio_client.get_bucket_policy('user-pics')

'{"Version":"2012-10-17","Statement":[{"Sid":"Statement1","Effect":"Allow","Principal":{"AWS":["*"]},"Action":["s3:GetObject"],"Resource":["arn:aws:s3:::user-pics/*"]}]}'

In [34]:
# https://awspolicygen.s3.amazonaws.com/policygen.html

policy = {
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "Statement1",
      "Effect": "Allow",
      "Principal": "*",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::user-pics/*"
    },
    {
      "Sid": "Statement2",
      "Effect": "Deny",
      "Principal": "*",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::user-pics/*",
      "Condition": {
        "StringEquals": {
          "s3:ExistingObjectTag/usergroup": "teacher"
        }
      }
    }
  ]
}

minio_client.set_bucket_policy('user-pics', json.dumps(policy))

In [21]:
minio_client.get_bucket_policy('user-pics')

'{"Version":"2012-10-17","Statement":[{"Sid":"Statement1","Effect":"Allow","Principal":{"AWS":["*"]},"Action":["s3:GetObject"],"Resource":["arn:aws:s3:::user-pics/*"]}]}'

---

In [24]:
from minio.commonconfig import SnowballObject

In [25]:
data.seek(0)
with open('dog.jpeg', 'rb') as f:
    data_dog = io.BytesIO(f.read())

minio_client.upload_snowball_objects(
    'user-pics',
    [
        SnowballObject('many_dog.jpeg', data=data_dog, length=len(data_dog.getvalue())),
        SnowballObject('many_cat.jpeg', data=data, length=len(data.getvalue()))
    ],
)

ObjectWriteResult(bucket_name='user-pics', object_name='snowball.0.6504242793488356.tar', version_id=None, etag='5b09d31004181c88fc4cd5523d381012', http_headers=HTTPHeaderDict({'Accept-Ranges': 'bytes', 'Content-Length': '0', 'ETag': '"5b09d31004181c88fc4cd5523d381012"', 'Server': 'MinIO', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Vary': 'Origin, Accept-Encoding', 'X-Amz-Id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'X-Amz-Request-Id': '186D6B6C20C22023', 'X-Content-Type-Options': 'nosniff', 'X-Ratelimit-Limit': '2024', 'X-Ratelimit-Remaining': '2024', 'X-Xss-Protection': '1; mode=block', 'Date': 'Sat, 11 Oct 2025 11:12:21 GMT'}), last_modified=None, location=None)

---

In [26]:
for i in minio_client.list_objects('user-pics'):
    print(i.object_name)

cat.jpg
cat_tags.jpg
many_cat.jpeg
many_dog.jpeg
data/


In [27]:
for i in minio_client.list_objects('user-pics'):
    minio_client.remove_object('user-pics', i.object_name)

In [28]:
for i in minio_client.list_objects('user-pics'):
    print(i.object_name)

data/


---

In [30]:
minio_client.remove_object('user-pics', 'data/cat_meta.jpg')

---

In [31]:
minio_client.remove_bucket('user-pics')

https://min.io/docs/minio/linux/developers/python/API.html